In [10]:
import os
import torch
import torch.nn as nn
from AIce.functions import testloader,redim
from  AIce.models import NNforNorms

device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')

weightlist=[]
for i in os.listdir('./weights'):
    """"Get the list of all the weights"""
    if i[-3:]=='.pt':
        weightlist.append(i)


data,_,_,means,stds,maxes,_=testloader('../data/DRIFT_DATA_TEST.csv',['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath','sin','cos', 'windnorm']
                                    ,'buoynorm',trainingset_loaded=False, training_file='../data/DRIFT_DATA_TRAIN.csv')# Get the means,stds,maxes using the longest list possible


inputstoweight={}

for k,weight in enumerate(weightlist):
    lst=[]
    
    with open(f'./weights/{weight[:-3]}.txt', 'r') as f:
        """Gets the inputlist associated with a particular Weight"""
        for i in range(2):
            f.readline()
        splits=f.read().split(':')
        lst=splits[1].strip()
        lst=lst.strip('[]').strip()
        lst=lst.split(',')
        lst=[x.strip()[1:-1] for x in lst]


    testdata,_,_,_,_,_,_=testloader('../data/DRIFT_DATA_TEST.csv',inputlist=lst
                                        ,target='buoynorm',means=means,stds=stds,maxes=maxes,trainingset_loaded=True)# Get the means,stds,maxes using the longest list possible

    mlp256= NNforNorms(len(lst)).to(device) # Creates a model for the specific n_inputs
    mlp256.load_state_dict(torch.load(f'./weights/{weight}', weights_only=True))# loads the weights onto the model
    mlp256.eval()  # if you're doing inference, not continuing training
    name=weight[11:19].strip('_') # I wanted a simpler name to differentiate them

    inputstoweight[name]=lst # saves the list of input into a dict
   # specificdf=data[lst] # gets the data for a specific inputlist

    out=mlp256(torch.tensor(testdata.drop(['buoynorm'],axis=1).values, dtype= torch.float32).to(device)).to('cpu')# runs the data into the model 
    data[name]=out.detach().numpy() # save the model prediction in the datapd with name associated to n_inputs



Bath size is the full test set
Target is buoynorm normalized by log1p
Sin and Cos added
Bathymetry (bath) normalized by maximum
x/y (x_EASE, y_EASE) normalized by maximum
Windnorm normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
Bath size is the full test set
Target is buoynorm normalized by log1p
Sin and Cos added
Bathymetry (bath) normalized by maximum
x/y (x_EASE, y_EASE) normalized by maximum
Windnorm normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
Bath size is the full test set
Target is buoynorm normalized by log1p
Wind components (u/v_ERA5) normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
Bath size is the full test set
Target is buoynorm normalized by log1p
Wind components (u/v_ERA5) normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
Bath size is the full test set
Target is buoynorm normalized by 

In [11]:
print(data.columns)

Index(['buoynorm', 'x_EASE', 'y_EASE', 'u_ERA5', 'v_ERA5', 'sic_CDR',
       'h_piomas', 'sin', 'cos', 'bath', 'windnorm', '10inputs', '2inputs',
       '3inputs', '4inputs', '5inputs', '6inputs', '7inputs', '9inputs'],
      dtype='object')
